## Simran Amesar

## Matriculation number - 100007050

In [8]:
texts = [
    "This phone is excellent",
    "Amazing battery life",
    "Very fast performance",
    "Camera quality is great",
    "I love this laptop",
    "Terrible customer service",
    "Battery drains quickly",
    "Very slow device",
    "Worst phone ever",
    "I hate this product",
    "I like this product but it's not the best"
]

In [9]:
# 1 = Positive
# 0 = Negative

labels = [1,1,1,1,1,0,0,0,0,0,1]

In [10]:
import numpy as np

from gensim.models import Word2Vec

In [11]:
tokenized_texts = [
    text.lower().split()
    for text in texts
]

In [12]:
print(tokenized_texts)

[['this', 'phone', 'is', 'excellent'], ['amazing', 'battery', 'life'], ['very', 'fast', 'performance'], ['camera', 'quality', 'is', 'great'], ['i', 'love', 'this', 'laptop'], ['terrible', 'customer', 'service'], ['battery', 'drains', 'quickly'], ['very', 'slow', 'device'], ['worst', 'phone', 'ever'], ['i', 'hate', 'this', 'product'], ['i', 'like', 'this', 'product', 'but', "it's", 'not', 'the', 'best']]


In [13]:
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4
)

In [14]:
print(w2v_model.wv.index_to_key)

['this', 'i', 'product', 'very', 'battery', 'is', 'phone', 'best', 'the', 'not', "it's", 'but', 'like', 'hate', 'ever', 'worst', 'device', 'slow', 'quickly', 'drains', 'service', 'customer', 'terrible', 'laptop', 'love', 'great', 'quality', 'camera', 'performance', 'fast', 'life', 'amazing', 'excellent']


In [15]:
print(w2v_model.wv['excellent'])

[ 0.00260033 -0.01960861  0.00917553 -0.00107645  0.01266419  0.00356695
 -0.0062596   0.01551995  0.00310933  0.00011042 -0.00922591 -0.01690705
 -0.01553366  0.01734102 -0.01784992  0.01806943 -0.01856204 -0.00055351
 -0.00381409 -0.01786229  0.01726012  0.01355563  0.00603888  0.00966691
  0.00022438  0.01884936  0.01404257 -0.01970745 -0.00886644 -0.00258022
  0.00609545 -0.0086479   0.00289833 -0.0156918   0.00555615  0.00940538
  0.00987463 -0.0063514  -0.01685408 -0.01844124 -0.0014458  -0.01465493
 -0.01362993  0.01224001  0.01434461  0.00423484 -0.0157988  -0.01139798
  0.01610369  0.00784169]


In [52]:
print(
    w2v_model.wv.most_similar('excellent')
)

[('worst', 0.3173593282699585), ('this', 0.24072179198265076), ('the', 0.2185668796300888), ('customer', 0.2177027314901352), ('like', 0.18460319936275482), ('life', 0.1783401072025299), ('ever', 0.17044849693775177), ('performance', 0.13190807402133942), ('great', 0.12935112416744232), ('slow', 0.11862964183092117)]


In [53]:
def sentence_vector(sentence, model):

    words = sentence.lower().split()

    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [54]:
X = np.array([
    sentence_vector(text, w2v_model)
    for text in texts
])

In [55]:
print(X.shape)

(11, 50)


In [56]:
import tensorflow as tf
from tensorflow.keras.models import Sequential # Builds the layers one by one
from tensorflow.keras.layers import Dense #creates a fully connected layer


In [57]:
model = Sequential() 
# Input layer + hidden layer
#  X.shape[1]: We set the input shape to the number of features (words) in our BoW representation
#8 = number of neurons in hidden layer
model.add(Dense(8,activation='relu',input_shape=(X.shape[1],)))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Show model architecture
model.summary()

# Train the model
history = model.fit(X, labels,
          epochs=100, #number of times the model will see the entire dataset
          verbose=0) #verbose=0 means no output during training


# Test the model with a new review
new_text = ["Amazing laptop"]

# Convert the new text to a numeric vector using the same vectorizer
new_vector = vectorizer.transform(new_text).toarray()

# Make a prediction
prediction = model.predict(new_vector)

print("\nProbability:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Positive Review")
else:
    print("Negative Review")

/Users/chaahatamesar/Desktop/NLP/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_22 (Dense)                │ (None, 8)              │           408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 417 (1.63 KB)

 Trainable params: 417 (1.63 KB)

 Non-trainable params: 0 (0.00 B)

ValueError: Unrecognized data type: x=[[-3.02010705e-03 -1.46007631e-03  1.20780454e-03 -6.50535687e-04
  -3.74480337e-03 -7.68226618e-03  2.11486733e-03  9.67220031e-03
  -1.93048059e-03 -7.46707385e-03  8.21980182e-03 -4.52732202e-03
   2.14464962e-05  5.25866402e-03 -6.28688140e-03  7.25233275e-03
   5.02849137e-03  4.26297629e-04 -1.25868283e-02 -2.86974944e-03
   1.08132232e-02  7.41903577e-03  1.17779039e-02  8.51637311e-03
  -3.23998998e-03  8.44161399e-03  1.09229684e-02  1.78938266e-03
  -6.38724957e-03 -2.39460589e-03 -1.28231326e-03 -8.76462460e-03
  -2.18801014e-03 -8.74208752e-03  2.80098664e-03  2.91802641e-03
   8.77284631e-03 -1.12164086e-02 -4.73015197e-03 -2.04461045e-03
  -5.69954701e-03  4.27276827e-03 -5.50635159e-05 -2.58941203e-03
   1.12169664e-02  5.01645543e-03 -3.40572256e-03  1.01977400e-03
   9.06084012e-03  1.88085693e-03]
 [-7.98817538e-03  2.56673805e-03  3.31444782e-03  9.79254488e-03
  -3.75279524e-05  5.73929027e-03 -1.02291780e-03  2.45272554e-03
  -